# Exploratory Data Analysis: Unified Dataset

This notebook analyzes the unified dataset containing math word problems and other text data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style('whitegrid')

In [ ]:
# Load the dataset
data_path = Path('data/collection/unified_dataset.jsonl')
df = pd.read_json(data_path, lines=True)
print(f"Dataset shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")

In [ ]:
# Preview the first few rows
display(df.head(3))

In [ ]:
# Missingness analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print("Missing values summary:")
display(missing_df)

In [ ]:
# Label distribution (if label column exists and has values)
if 'label' in df.columns and df['label'].notna().sum() > 0:
    label_counts = df['label'].value_counts()
    print(f"Label column: {df['label'].notna().sum()} non-null values out of {len(df)}")
    print(f"Unique labels: {df['label'].nunique()}")
    fig, ax = plt.subplots(figsize=(10, 4))
    label_counts.head(20).plot(kind='bar', ax=ax)
    ax.set_title('Label Distribution (Top 20)')
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No label column or all labels are null.")

In [ ]:
# Text length distribution (if text column exists)
if 'text' in df.columns and df['text'].notna().sum() > 0:
    df['text_length'] = df['text'].str.len()
    df['word_count'] = df['text'].str.split().str.len()
    print("Text length statistics (characters):")
    print(df['text_length'].describe())
    print("\nWord count statistics:")
    print(df['word_count'].describe())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df['text_length'].hist(bins=30, ax=axes[0])
    axes[0].set_title('Text Length Distribution (Characters)')
    axes[0].set_xlabel('Characters')
    df['word_count'].hist(bins=30, ax=axes[1])
    axes[1].set_title('Word Count Distribution')
    axes[1].set_xlabel('Words')
    plt.tight_layout()
    plt.show()
else:
    print("No text column or all text is null.")

In [ ]:
# Source distribution
if 'source' in df.columns:
    source_counts = df['source'].value_counts()
    print("Source distribution:")
    display(source_counts)
    fig, ax = plt.subplots(figsize=(8, 4))
    source_counts.plot(kind='bar', ax=ax)
    ax.set_title('Source Distribution')
    ax.set_xlabel('Source')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No source column found.")

In [ ]:
# Metadata inspection
if 'metadata' in df.columns and df['metadata'].notna().sum() > 0:
    # Extract metadata keys
    metadata_keys = df['metadata'].dropna().apply(lambda x: list(x.keys()) if isinstance(x, dict) else []).explode().unique()
    print(f"Metadata keys found: {list(metadata_keys)}")
    if 'cleaning_status' in metadata_keys:
        cleaning_counts = df['metadata'].apply(lambda x: x.get('cleaning_status') if isinstance(x, dict) else None).value_counts()
        print("\nCleaning status distribution:")
        display(cleaning_counts)
else:
    print("No metadata column or all metadata is null.")

## Data Quality Review

### Analyzer View

- Task interpretation: unknown
- Primary modality: text
- Target semantics: unknown
- Relevant checks: text_non_empty - all 124 rows have valid text, label_format_validity - labels follow CoT format with '#### <answer>', source_distribution - tracks provenance across 3 sources, metadata_completeness - metadata present for all rows
- Lower-value checks: class_balance_plots - not applicable; each label is unique (each math problem has unique solution), numeric_outlier_detection - 15 text length 'outliers' are legitimate long problems, not errors, audio_image_modality_checks - these columns are entirely null and should be dropped
- Priority actions: drop_unused_columns_audio_image, preserve_all_rows_including_unlabeled, flag_unlabeled_rows_by_source

### Strategy Justification

This is a math reasoning dataset where all 124 rows contain valid problem text. The retention-oriented strategy preserves all data including 24 unlabeled rows (project-euler + all-russian) which are valuable for inference or future label generation. Dropping audio/image columns removes entirely null/unused data. No deduplication needed as each row is unique. This maximizes data utility for both supervised training and inference evaluation.

- Missing values: `N/A - text dataset, no numeric imputation needed`
- Duplicates: `drop - no duplicates found after normalized text check`
- Outliers: `N/A - text dataset, no numeric outlier clipping needed`

### Findings

- Missing values before cleaning: 262
- Duplicate rows before cleaning: 0
- Numeric outliers before cleaning: 15
- Imbalance column: `label`
- Majority class share after cleaning: not applicable

### Before / After

- Missing values: 0 -> 0
- Duplicates: 0 -> 0
- Outliers: 0 -> 0


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

raw_df = pd.read_json('data/collection/unified_dataset.jsonl', lines=True)
clean_df = pd.read_json('data/quality/cleaned_dataset.jsonl', lines=True)
primary_modality = 'text'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
missing_counts = raw_df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]
if not missing_counts.empty:
    sns.barplot(x=missing_counts.values, y=missing_counts.index, ax=axes[0, 0], color='#d97706')
    axes[0, 0].set_title('Missing values by column')
else:
    axes[0, 0].text(0.5, 0.5, 'No missing values', ha='center', va='center')
    axes[0, 0].set_axis_off()

if 'text' in raw_df.columns:
    text_lengths = raw_df['text'].fillna('').astype(str).str.split().str.len()
    sns.histplot(text_lengths, bins=30, ax=axes[0, 1], color='#2563eb')
    axes[0, 1].set_title('Text length distribution')
else:
    axes[0, 1].text(0.5, 0.5, 'No text column', ha='center', va='center')
    axes[0, 1].set_axis_off()

numeric_columns = raw_df.select_dtypes(include=['number']).columns.tolist()
if numeric_columns and not (len(numeric_columns) == 1 and 'label' in numeric_columns and primary_modality == 'text'):
    sns.boxplot(data=raw_df[numeric_columns], orient='h', ax=axes[1, 0], color='#f59e0b')
    axes[1, 0].set_title('Raw numeric distributions')
    sns.boxplot(data=clean_df[numeric_columns], orient='h', ax=axes[1, 1], color='#10b981')
    axes[1, 1].set_title('Cleaned numeric distributions')
else:
    if 'text' in raw_df.columns:
        raw_lengths = raw_df['text'].fillna('').astype(str).str.split().str.len()
        clean_lengths = clean_df['text'].fillna('').astype(str).str.split().str.len()
        sns.histplot(raw_lengths, bins=30, ax=axes[1, 0], color='#f59e0b')
        axes[1, 0].set_title('Raw text length distribution')
        sns.histplot(clean_lengths, bins=30, ax=axes[1, 1], color='#10b981')
        axes[1, 1].set_title('Cleaned text length distribution')
    else:
        axes[1, 0].text(0.5, 0.5, 'No numeric columns', ha='center', va='center')
        axes[1, 0].set_axis_off()
        axes[1, 1].text(0.5, 0.5, 'No numeric columns', ha='center', va='center')
        axes[1, 1].set_axis_off()

plt.tight_layout()
plt.show()
